# LC 141 — Linked List Cycle
**Day 43 | Pattern: Floyd's Tortoise and Hare | Difficulty: Easy**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Use two pointers at different speeds.
The slow pointer moves one step at a time; the fast pointer moves
two. If there is a cycle, fast will lap slow and they will meet.
If there is no cycle, fast will hit <code>None</code>. O(1) space.
</div>

## Official Problem Statement

Given `head`, the head of a linked list, determine if the linked
list has a cycle in it.

There is a cycle in a linked list if there is some node in the
list that can be reached again by continuously following the
`next` pointer.

Return `true` if there is a cycle in the linked list. Otherwise,
return `false`.

**Constraints:**
- The number of nodes is in the range `[0, 10^4]`.
- `-10^5 <= Node.val <= 10^5`
- `pos` is `-1` or a valid index in the linked list
  (`pos` is only used internally to create the cycle).

**Follow up:** Can you solve it using O(1) memory?

## What This Is Actually Asking

A normal linked list ends when a node's `next` is `None`. A cyclic
list has one node pointing back to an earlier node, creating an
infinite loop. You need to detect this without using a hash set
(which would take O(n) space). The classic solution uses two
pointers moving at different speeds — if a cycle exists, the
faster one will inevitably catch the slower one.

## Walk Through an Example by Hand

```
List: 3 -> 2 -> 0 -> -4
                      |
                      v
              (back to node 2, pos=1)
```

`slow=3, fast=3`

**Step 1:** `slow=2, fast=0`
  (slow: 3->2, fast: 3->2->0)

**Step 2:** `slow=0, fast=2`
  (slow: 2->0, fast: 0->-4->2)

**Step 3:** `slow=-4, fast=-4`  <- MEET!
  (slow: 0->-4, fast: 2->0->-4)

`slow == fast` → return `True`

---
No-cycle example: `1 -> 2 -> None`

`slow=1, fast=1`

**Step 1:** `slow=2, fast=None` → `fast` is None → return `False`

## The Picture

```
NO CYCLE:

  slow  fast
   |     |
  [1] -> [2] -> [3] -> [4] -> None

  fast reaches None first -> return False

WITH CYCLE:

  [3] -> [2] -> [0] -> [-4]
          ^              |
          |______________|   (pos=1)

  slow moves 1 step, fast moves 2 steps:

  Round 1:  slow=[2]   fast=[0]
  Round 2:  slow=[0]   fast=[2]  (fast lapped around)
  Round 3:  slow=[-4]  fast=[-4] <- COLLISION!

  They meet inside the cycle -> return True

KEY: In a cycle, fast gains 1 node per round on slow.
     They MUST meet within cycle_length steps.
```

## When To Use This Pattern

- When asked to **detect a cycle** in a linked list, think
  **Floyd's slow/fast pointer**.
- When a problem involves **finding the middle** of a linked
  list, think **slow moves 1, fast moves 2**.
- When you need **O(1) space** to detect repetition in a
  sequence, think **two-speed pointers**.
- When asked to **find the entry point** of a cycle (LC 142),
  think **reset one pointer to head after meeting, then both
  move 1 step until they meet again**.
- When dealing with **Happy Number (LC 202)** or any sequence
  that might loop, think **tortoise and hare**.

## The Approach

Start both `slow` and `fast` at `head`. In each iteration, advance
`slow` by one node and `fast` by two nodes. Before each move,
check that `fast` and `fast.next` are not `None` — if either is,
there is no cycle and you return `False`. If at any point
`slow == fast`, the pointers have met inside a cycle and you
return `True`.

In [ ]:
from typing import Optional


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def make_list(vals):
    dummy = ListNode(0)
    cur = dummy
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next


def to_list(head):
    result = []
    while head:
        result.append(head.val)
        head = head.next
    return result


def make_list_with_cycle(vals, pos):
    # pos=-1 means no cycle
    nodes = [ListNode(v) for v in vals]
    for i in range(len(nodes) - 1):
        nodes[i].next = nodes[i + 1]
    if pos >= 0:
        nodes[-1].next = nodes[pos]
    return nodes[0] if nodes else None

In [ ]:
def test_harness(func):
    cases = [
        # (vals, pos, expected)
        # pos=-1 means no cycle; pos>=0 is cycle entry index
        ([3, 2, 0, -4], 1,  True),   # cycle at index 1
        ([1, 2],        0,  True),   # cycle at index 0
        ([1],           -1, False),  # single node, no cycle
        ([],            -1, False),  # empty list
        ([1, 2, 3, 4],  -1, False),  # no cycle
    ]
    passed = 0
    for i, (vals, pos, expected) in enumerate(cases):
        if vals:
            head = make_list_with_cycle(vals, pos)
        else:
            head = None
        result = func(head)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"vals={vals} pos={pos} | "
            f"got={result} | "
            f"expected={expected}"
        )
    print(f"\nResult: {passed}/{len(cases)} passed")

In [ ]:
def hasCycle(head: Optional[ListNode]) -> bool:
    """
    Detect if a linked list contains a cycle.

    Strategy:
        Floyd's Tortoise and Hare.
        slow = head, fast = head.
        While fast and fast.next are not None:
          slow = slow.next
          fast = fast.next.next
          if slow == fast: return True
        return False

    Args:
        head: Head node of the linked list.

    Returns:
        True if cycle detected, False otherwise.

    Time:  O(n)
    Space: O(1)
    """
    # --- debug prints (remove before submit) ---
    print(f"head is None: {head is None}")
    print(f"Starting slow/fast traversal...")
    # -------------------------------------------

    pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(hasCycle)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute force (hash set of visited nodes) | O(n) | O(n) | Easy but uses extra memory |
| Floyd's slow/fast pointer | O(n) | O(1) | Optimal |
| Mark visited (mutate node vals) | O(n) | O(1) | Destructive, not safe |

## Real World Connection

At Citi, detecting circular dependencies in settlement chains
or workflow graphs is a real operational risk concern — a cycle
in a processing pipeline means a transaction could loop forever.
In AWS Step Functions or event-driven architectures, detecting
circular event routing before it causes an infinite loop is
critical. As a Data Engineer, the two-pointer technique is also
used in stream processing to find repeated patterns in a data
stream with constant memory, making it valuable for monitoring
pipelines and anomaly detection.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra